<a href="https://colab.research.google.com/github/aakarshityadav0-gif/operating-system/blob/main/os.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Program 1:**
Introduction to Linux System Calls, Process Creation using fork(), exec() and wait()


In [ ]:
# AAKARSHIT YADAV 2472121
import os

def main():
    parent_id = os.getpid()
    print(f"Parent process initialised, PID = {parent_id}")

    child_pid = os.fork()

    if child_pid == 0:
        current_id = os.getpid()
        parent_id = os.getppid()
        print(f"\nChild process running: PID = {current_id}, Parent PID = {parent_id}")
        print("Child: Invoking 'date' command via exec()")
        os.execvp("date", ["date"])
        print("This code block is unreachable")
    else:
        print(f"Parent: Successfully created child with PID = {child_pid}")
        terminated_pid, termination_status = os.wait()
        exit_code = os.WEXITSTATUS(termination_status)
        print(f"\nParent: Child process {terminated_pid} terminated")
        print(f"Parent: Exit status of child = {exit_code}")
        print("Parent execution finished.")

if __name__ == '__main__':
    main()

Parent process initialised, PID = 541

Child process running: PID = 566, Parent PID = 541
Child: Invoking 'date' command via exec()
Parent process initialised, PID = 541
Parent: Successfully created child with PID = 566

Parent: Child process 566 terminated
Parent: Exit status of child = 0
Parent execution finished.


/tmp/ipykernel_541/3800504660.py:8: DeprecationWarning: This process (pid=541) is multi-threaded, use of fork() may lead to deadlocks in the child.
  child_pid = os.fork()


**Program 2:**
Thread Creation and Management using POSIX Threads (pthreads)


In [ ]:
# AAKARSHIT YADAV 2472121
import threading
import time

thread_outputs = {}

def task_handler(tid):
    curr_name = threading.current_thread().name
    print(f"[Thread {tid}] started execution (Name: {curr_name})")
    time.sleep(1.0)  # Simulating processing delay
    print(f"[Thread {tid}] finished task")
    thread_outputs[tid] = tid * 10

def run_multithreading():
    total_threads = 3
    active_threads = []

    print(f"Main Thread: Spawning {total_threads} worker threads...")

    for idx in range(1, total_threads + 1):
        worker_thread = threading.Thread(
            target=task_handler,
            args=(idx,),
            name=f"Worker-{idx}"
        )
        active_threads.append(worker_thread)
        worker_thread.start()

    for worker_thread in active_threads:
        worker_thread.join()

    for tid, res in thread_outputs.items():
        print(f"Main Thread: Thread {tid} returned value {res}")

    print("Main Thread: All background execution finalized")

if __name__ == '__main__':
    run_multithreading()

Main Thread: Spawning 3 worker threads...
[Thread 1] started execution (Name: Worker-1)
[Thread 2] started execution (Name: Worker-2)
[Thread 3] started execution (Name: Worker-3)
[Thread 1] finished task
[Thread 2] finished task
[Thread 3] finished task
Main Thread: Thread 1 returned value 10
Main Thread: Thread 2 returned value 20
Main Thread: Thread 3 returned value 30
Main Thread: All background execution finalized


**Program 3:**
CPU Scheduling Algorithms — FCFS, SJF, Priority and Round Robin


In [ ]:
# AAKARSHIT YADAV 2472121

def show_metrics(processes):
    total_processes = len(processes)
    cumulative_waiting_time = 0
    cumulative_turnaround_time = 0

    print("\nPID\tArrival\tBurst\tComp\tTAT\tWT")
    for p in processes:
        print(f"{p['pid']}\t{p['at']}\t{p['bt']}\t{p['ct']}\t{p['tat']}\t{p['wt']}")
        cumulative_waiting_time += p['wt']
        cumulative_turnaround_time += p['tat']

    print(f"Average Waiting Time    = {cumulative_waiting_time / total_processes:.2f}")
    print(f"Average Turnaround Time = {cumulative_turnaround_time / total_processes:.2f}")

# First-Come, First-Served Scheduler
def run_fcfs(job_queue):
    current_time = 0
    for job in job_queue:
        if current_time < job['at']:
            current_time = job['at']
        current_time += job['bt']
        job['ct'] = current_time
        job['tat'] = job['ct'] - job['at']
        job['wt'] = job['tat'] - job['bt']
    show_metrics(job_queue)

# Shortest Job First Scheduler (Non-preemptive)
def run_sjf(job_queue):
    num_jobs = len(job_queue)
    completed_jobs = 0
    current_time = 0
    is_completed = [False] * num_jobs

    while completed_jobs < num_jobs:
        selected_idx = -1
        shortest_burst = float('inf')

        for i in range(num_jobs):
            if not is_completed[i] and job_queue[i]['at'] <= current_time:
                if job_queue[i]['bt'] < shortest_burst:
                    shortest_burst = job_queue[i]['bt']
                    selected_idx = i

        if selected_idx == -1:
            current_time += 1
            continue

        current_time += job_queue[selected_idx]['bt']
        job_queue[selected_idx]['ct'] = current_time
        job_queue[selected_idx]['tat'] = job_queue[selected_idx]['ct'] - job_queue[selected_idx]['at']
        job_queue[selected_idx]['wt'] = job_queue[selected_idx]['tat'] - job_queue[selected_idx]['bt']
        is_completed[selected_idx] = True
        completed_jobs += 1

    show_metrics(job_queue)

# Non-preemptive Priority Scheduler (Lower number -> higher priority)
def run_priority(job_queue):
    num_jobs = len(job_queue)
    completed_jobs = 0
    current_time = 0
    is_completed = [False] * num_jobs

    while completed_jobs < num_jobs:
        selected_idx = -1
        highest_priority = float('inf')

        for i in range(num_jobs):
            if not is_completed[i] and job_queue[i]['at'] <= current_time:
                if job_queue[i]['pr'] < highest_priority:
                    highest_priority = job_queue[i]['pr']
                    selected_idx = i

        if selected_idx == -1:
            current_time += 1
            continue

        current_time += job_queue[selected_idx]['bt']
        job_queue[selected_idx]['ct'] = current_time
        job_queue[selected_idx]['tat'] = job_queue[selected_idx]['ct'] - job_queue[selected_idx]['at']
        job_queue[selected_idx]['wt'] = job_queue[selected_idx]['tat'] - job_queue[selected_idx]['bt']
        is_completed[selected_idx] = True
        completed_jobs += 1

    show_metrics(job_queue)

# Round Robin Scheduler
def run_round_robin(job_queue, quantum):
    num_jobs = len(job_queue)
    current_time = 0
    completed_jobs = 0

    for job in job_queue:
        job['rem'] = job['bt']

    ready_queue = []
    in_queue_flag = [False] * num_jobs

    for i in range(num_jobs):
        if job_queue[i]['at'] <= current_time:
            ready_queue.append(i)
            in_queue_flag[i] = True

    queue_ptr = 0

    while completed_jobs < num_jobs:
        if queue_ptr == len(ready_queue):
            current_time += 1
            for i in range(num_jobs):
                if not in_queue_flag[i] and job_queue[i]['rem'] > 0 and job_queue[i]['at'] <= current_time:
                    ready_queue.append(i)
                    in_queue_flag[i] = True
            continue

        active_idx = ready_queue[queue_ptr]
        queue_ptr += 1

        execution_slice = min(job_queue[active_idx]['rem'], quantum)
        current_time += execution_slice
        job_queue[active_idx]['rem'] -= execution_slice

        for i in range(num_jobs):
            if not in_queue_flag[i] and job_queue[i]['rem'] > 0 and job_queue[i]['at'] <= current_time and i != active_idx:
                ready_queue.append(i)
                in_queue_flag[i] = True

        if job_queue[active_idx]['rem'] > 0:
            ready_queue.append(active_idx)
        else:
            job_queue[active_idx]['ct'] = current_time
            job_queue[active_idx]['tat'] = job_queue[active_idx]['ct'] - job_queue[active_idx]['at']
            job_queue[active_idx]['wt'] = job_queue[active_idx]['tat'] - job_queue[active_idx]['bt']
            completed_jobs += 1
            in_queue_flag[active_idx] = False

    show_metrics(job_queue)

if __name__ == "__main__":
    n_jobs = int(input("Enter total number of processes: "))
    processes = []

    for k in range(n_jobs):
        print(f"\nInput parameters for Process {k + 1}:")
        process_id = int(input("ID: "))
        arrival_time = int(input("Arrival Time: "))
        burst_time = int(input("Burst Time: "))
        priority_val = int(input("Priority Value: "))

        processes.append({
            'pid': process_id,
            'at': arrival_time,
            'bt': burst_time,
            'pr': priority_val,
            'ct': 0,
            'tat': 0,
            'wt': 0,
            'rem': 0
        })

    print("\nSelect a CPU Scheduling Algorithm:")
    print("1 -> First Come First Serve (FCFS)")
    print("2 -> Shortest Job First (SJF)")
    print("3 -> Priority Scheduling")
    print("4 -> Round Robin (RR)")

    algo_choice = int(input("Enter Choice ID (1-4): "))

    if algo_choice == 1:
        run_fcfs([dict(p) for p in processes])
    elif algo_choice == 2:
        run_sjf([dict(p) for p in processes])
    elif algo_choice == 3:
        run_priority([dict(p) for p in processes])
    elif algo_choice == 4:
        q_val = int(input("Enter Time Quantum: "))
        if q_val <= 0:
            print("Error: Quantum must be positive.")
        else:
            run_round_robin([dict(p) for p in processes], q_val)
    else:
        print("Invalid Selection!")

**Program 4**
Process Synchronization — Producer-Consumer Problem using Semaphores


In [ ]:
# AAKARSHIT YADAV 2472121
import threading
import time

QUEUE_CAPACITY = 5
TOTAL_TRANSFERS = 10

data_buffer = [0] * QUEUE_CAPACITY
write_ptr = 0
read_ptr = 0

# Coordination Semaphores
slots_available = threading.Semaphore(QUEUE_CAPACITY)
items_available = threading.Semaphore(0)
lock_mutex = threading.Semaphore(1)

def item_producer():
    global write_ptr
    for item_val in range(1, TOTAL_TRANSFERS + 1):
        slots_available.acquire()
        lock_mutex.acquire()

        data_buffer[write_ptr] = item_val
        print(f"Producer: Inserted item {item_val} into index {write_ptr}")
        write_ptr = (write_ptr + 1) % QUEUE_CAPACITY

        lock_mutex.release()
        items_available.release()

        time.sleep(0.1)

def item_consumer():
    global read_ptr
    for _ in range(1, TOTAL_TRANSFERS + 1):
        items_available.acquire()
        lock_mutex.acquire()

        extracted_item = data_buffer[read_ptr]
        print(f"\t\tConsumer: Extracted item {extracted_item} from index {read_ptr}")
        read_ptr = (read_ptr + 1) % QUEUE_CAPACITY

        lock_mutex.release()
        slots_available.release()

        time.sleep(0.15)

def run_pipeline():
    producer_thread = threading.Thread(target=item_producer)
    consumer_thread = threading.Thread(target=item_consumer)

    producer_thread.start()
    consumer_thread.start()

    producer_thread.join()
    consumer_thread.join()

    print("Production-Consumption cycle completed successfully.")

if __name__ == "__main__":
    run_pipeline()

Producer produced item 1 at slot 0
		Consumer consumed item 1 from slot 0
Producer produced item 2 at slot 1
		Consumer consumed item 2 from slot 1
Producer produced item 3 at slot 2
Producer produced item 4 at slot 3
		Consumer consumed item 3 from slot 2
Producer produced item 5 at slot 4
		Consumer consumed item 4 from slot 3
Producer produced item 6 at slot 0
Producer produced item 7 at slot 1
		Consumer consumed item 5 from slot 4
Producer produced item 8 at slot 2
		Consumer consumed item 6 from slot 0
Producer produced item 9 at slot 3
Producer produced item 10 at slot 4
		Consumer consumed item 7 from slot 1
		Consumer consumed item 8 from slot 2
		Consumer consumed item 9 from slot 3
		Consumer consumed item 10 from slot 4
All items produced and consumed.


**Program 5:**
Process Synchronization — Readers-Writers Problem using Semaphores


In [ ]:
# AAKARSHIT YADAV 2472121
import threading
import time

# Synchronization Primitives
mutex_lock = threading.Semaphore(1)
write_permission = threading.Semaphore(1)

active_readers_count = 0
database_value = 0

def execute_reader(reader_id):
    global active_readers_count

    # Lock the control block to update reader count safely
    mutex_lock.acquire()
    active_readers_count += 1
    if active_readers_count == 1:
        write_permission.acquire()  # Block writers if this is the first reader
    mutex_lock.release()

    # Perform read operation
    print(f"Reader {reader_id}: Read current value = {database_value}")
    time.sleep(0.2)

    # Exit code block
    mutex_lock.acquire()
    active_readers_count -= 1
    if active_readers_count == 0:
        write_permission.release()  # Last reader unlocks access for writers
    mutex_lock.release()

def execute_writer(writer_id):
    global database_value

    write_permission.acquire()  # Lock resource exclusively
    database_value += 1
    print(f"Writer {writer_id}: Updated value to = {database_value}")
    time.sleep(0.3)
    write_permission.release()  # Unlock resource

def run_readers_writers():
    reader_threads = []
    writer_threads = []

    # Start first Writer
    w1 = threading.Thread(target=execute_writer, args=(1,))
    writer_threads.append(w1)
    w1.start()

    # Spawn 5 concurrent Readers
    for idx in range(1, 6):
        r_thread = threading.Thread(target=execute_reader, args=(idx,))
        reader_threads.append(r_thread)
        r_thread.start()

    # Start second Writer
    w2 = threading.Thread(target=execute_writer, args=(2,))
    writer_threads.append(w2)
    w2.start()

    for r_thread in reader_threads:
        r_thread.join()
    for w_thread in writer_threads:
        w_thread.join()

    print("Simulation cycle of Readers-Writers finished.")

if __name__ == "__main__":
    run_readers_writers()

Writer 1: wrote shared_data = 1
Reader 1: read shared_data = 1
Reader 2: read shared_data = 1
Reader 3: read shared_data = 1
Reader 4: read shared_data = 1
Reader 5: read shared_data = 1
Writer 2: wrote shared_data = 2
All readers and writers have finished.


**Program 6:**
Dining Philosophers Problem using Semaphores


In [ ]:
# AAKARSHIT YADAV 2472121
import threading
import time

PHILOSOPHERS_COUNT = 5

# Initialize semaphores for resources (forks)
chopsticks = [threading.Semaphore(1) for _ in range(PHILOSOPHERS_COUNT)]

# Room limit controller to prevent deadlock (allowing at most N-1 philosophers in the dining area)
dining_room_limit = threading.Semaphore(PHILOSOPHERS_COUNT - 1)

def start_philosopher(phil_id):
    left_chopstick = phil_id
    right_chopstick = (phil_id + 1) % PHILOSOPHERS_COUNT

    for cycle in range(2):
        print(f"Philosopher {phil_id} is contemplating")
        time.sleep(0.1)

        # Access dining room
        dining_room_limit.acquire()

        # Pick up chopsticks
        chopsticks[left_chopstick].acquire()
        chopsticks[right_chopstick].acquire()

        print(f"Philosopher {phil_id} is eating using forks {left_chopstick} and {right_chopstick}")
        time.sleep(0.15)

        # Return chopsticks
        chopsticks[right_chopstick].release()
        chopsticks[left_chopstick].release()

        # Exit dining room
        dining_room_limit.release()

        print(f"Philosopher {phil_id} completed eating phase and returned forks")

def run_simulation():
    threads = []
    for index in range(PHILOSOPHERS_COUNT):
        phil_thread = threading.Thread(target=start_philosopher, args=(index,))
        threads.append(phil_thread)
        phil_thread.start()

    for phil_thread in threads:
        phil_thread.join()

    print("All Dining Philosopher cycles concluded.")

if __name__ == "__main__":
    run_simulation()

Philosopher 0 is thinking
Philosopher 1 is thinking
Philosopher 2 is thinking
Philosopher 3 is thinking
Philosopher 4 is thinking
Philosopher 0 is eating (fork 0 and fork 1)
Philosopher 2 is eating (fork 2 and fork 3)
Philosopher 0 finished eating and put down forks
Philosopher 0 is thinking
Philosopher 4 is eating (fork 4 and fork 0)
Philosopher 2 finished eating and put down forks
Philosopher 2 is thinking
Philosopher 1 is eating (fork 1 and fork 2)
Philosopher 4 finished eating and put down forks
Philosopher 4 is thinking
Philosopher 3 is eating (fork 3 and fork 4)
Philosopher 1 finished eating and put down forks
Philosopher 1 is thinking
Philosopher 0 is eating (fork 0 and fork 1)
Philosopher 3 finished eating and put down forksPhilosopher 2 is eating (fork 2 and fork 3)
Philosopher 0 finished eating and put down forks
Philosopher 4 is eating (fork 4 and fork 0)

Philosopher 3 is thinking
Philosopher 2 finished eating and put down forks
Philosopher 1 is eating (fork 1 and fork 2)
P

**Program 7:**
Deadlock Avoidance using Banker's Algorithm


In [ ]:
# AAKARSHIT YADAV 2472121
NUM_PROCESSES = 5
NUM_RESOURCES = 3

avail_vector = [0] * NUM_RESOURCES
max_matrix = [[0] * NUM_RESOURCES for _ in range(NUM_PROCESSES)]
alloc_matrix = [[0] * NUM_RESOURCES for _ in range(NUM_PROCESSES)]
need_matrix = [[0] * NUM_RESOURCES for _ in range(NUM_PROCESSES)]

def compute_need_map():
    for r_idx in range(NUM_PROCESSES):
        for c_idx in range(NUM_RESOURCES):
            need_matrix[r_idx][c_idx] = max_matrix[r_idx][c_idx] - alloc_matrix[r_idx][c_idx]

def verify_safety():
    work_vector = avail_vector.copy()
    is_finished = [False] * NUM_PROCESSES
    safe_order = []
    processed_count = 0

    while processed_count < NUM_PROCESSES:
        allocation_found = False

        for p_idx in range(NUM_PROCESSES):
            if not is_finished[p_idx]:
                res_idx = 0
                while res_idx < NUM_RESOURCES and need_matrix[p_idx][res_idx] <= work_vector[res_idx]:
                    res_idx += 1

                if res_idx == NUM_RESOURCES:
                    for r_dim in range(NUM_RESOURCES):
                        work_vector[r_dim] += alloc_matrix[p_idx][r_dim]

                    safe_order.append(p_idx)
                    is_finished[p_idx] = True
                    processed_count += 1
                    allocation_found = True

        if not allocation_found:
            return False, []

    return True, safe_order

def process_allocation_request(process_id, request_vector):
    if process_id < 0 or process_id >= NUM_PROCESSES:
        print("Error: Process identifier out of bounds.")
        return False

    for res in range(NUM_RESOURCES):
        if request_vector[res] > need_matrix[process_id][res]:
            print("Execution halted: Request exceeds maximum declared demand.")
            return False

        if request_vector[res] > avail_vector[res]:
            print("Request Denied: Exceeds currently available system capacity.")
            return False

    # Speculatively allocate values
    for res in range(NUM_RESOURCES):
        avail_vector[res] -= request_vector[res]
        alloc_matrix[process_id][res] += request_vector[res]
        need_matrix[process_id][res] -= request_vector[res]

    is_safe_state, process_sequence = verify_safety()

    if is_safe_state:
        print("Allocation approved successfully.")
        print("Safe transition path:", " -> ".join(f"P{id}" for id in process_sequence))
        return True
    else:
        # Roll back changes on unsafe detection
        for res in range(NUM_RESOURCES):
            avail_vector[res] += request_vector[res]
            alloc_matrix[process_id][res] -= request_vector[res]
            need_matrix[process_id][res] += request_vector[res]

        print("Allocation Denied: This distribution will transition the environment into an unsafe state.")
        return False

def main():
    global avail_vector, max_matrix, alloc_matrix
    print(f"Provide current system available units ({NUM_RESOURCES} dimensions):")
    avail_vector = list(map(int, input().split()))

    if len(avail_vector) != NUM_RESOURCES or any(val < 0 for val in avail_vector):
        print("Error: Input values are invalid.")
        return

    print(f"Define Maximum Matrix ({NUM_PROCESSES} rows x {NUM_RESOURCES} columns):")
    for row in range(NUM_PROCESSES):
        max_matrix[row] = list(map(int, input(f"P{row}: ").split()))
        if len(max_matrix[row]) != NUM_RESOURCES or any(val < 0 for val in max_matrix[row]):
            print("Error: Row contains invalid size or value properties.")
            return

    print("Define Allocation Matrix:")
    for row in range(NUM_PROCESSES):
        alloc_matrix[row] = list(map(int, input(f"P{row}: ").split()))
        if len(alloc_matrix[row]) != NUM_RESOURCES or any(val < 0 for val in alloc_matrix[row]):
            print("Error: Row contains invalid size or value properties.")
            return

    for row in range(NUM_PROCESSES):
        for col in range(NUM_RESOURCES):
            if alloc_matrix[row][col] > max_matrix[row][col]:
                print("Constraint violated: Allocation cannot surpass declared upper limit.")
                return

    compute_need_map()

    print("\nComputed System Need Matrix:")
    for idx in range(NUM_PROCESSES):
        print(f"P{idx}: {need_matrix[idx]}")

    safe, seq = verify_safety()
    if safe:
        print("\nSystem Status: SAFE State verified.")
        print("Safe evaluation sequence:", " ".join(f"P{i}" for i in seq))
    else:
        print("\nSystem Status: UNSAFE State detected.")

    try:
        target_process = int(input("\nEnter target process initiating request (0-4): "))
        incoming_request = list(map(int, input(f"Enter demand vector ({NUM_RESOURCES} values): ").split()))

        if len(incoming_request) != NUM_RESOURCES or any(val < 0 for val in incoming_request):
            print("Error: Demand matrix format mismatch.")
            return

        if not safe:
            print("Execution blocked: Request rejected because the system is already unsafe.")
            return

        process_allocation_request(target_process, incoming_request)
    except ValueError:
        print("Execution stopped: Only numeric inputs allowed.")

if __name__ == "__main__":
    main()

Enter available resources (3 values):
3 3 2
Enter Maximum matrix (5 processes x 3 resources):
P0: 7 5 3
P1: 3 2 2
P2: 9 0 2
P3: 2 2 2
P4: 4 3 3
Enter Allocation matrix:
P0: 0 1 0
P1: 2 0 0
P2: 3 0 2
P3: 2 1 1
P4: 0 2 2

Need Matrix:
P0: [7, 4, 3]
P1: [1, 2, 2]
P2: [6, 0, 0]
P3: [0, 1, 1]
P4: [4, 1, 1]

System is in a SAFE state.
Safe sequence: P1 P3 P4 P0 P2

Enter process number requesting resources (0-4): 1
Enter request vector (3 values): 1 0 2
Request can be granted safely.
Safe sequence: P1 P3 P4 P0 P2


**Program 8:**
Contiguous Memory Allocation — First Fit, Best Fit and Worst Fit


In [ ]:
# AAKARSHIT YADAV 2472121
UPPER_LIMIT = 20

def run_first_fit(blocks, processes):
    num_blocks = len(blocks)
    num_proc = len(processes)

    current_mem = blocks.copy()
    allocation_map = [-1] * num_proc

    for p_idx in range(num_proc):
        for b_idx in range(num_blocks):
            if current_mem[b_idx] >= processes[p_idx]:
                allocation_map[p_idx] = b_idx
                current_mem[b_idx] -= processes[p_idx]
                break

    print("\n=== First Fit Allocation ===")
    print("Process ID\tProcess Size\tBlock Index")
    for idx in range(num_proc):
        dest = f"B{allocation_map[idx] + 1}" if allocation_map[idx] != -1 else "Unallocated"
        print(f"P{idx + 1}\t\t{processes[idx]}\t\t{dest}")

def run_best_fit(blocks, processes):
    num_blocks = len(blocks)
    num_proc = len(processes)

    current_mem = blocks.copy()
    allocation_map = [-1] * num_proc

    for p_idx in range(num_proc):
        selected_block = -1
        for b_idx in range(num_blocks):
            if current_mem[b_idx] >= processes[p_idx]:
                if selected_block == -1 or current_mem[b_idx] < current_mem[selected_block]:
                    selected_block = b_idx

        if selected_block != -1:
            allocation_map[p_idx] = selected_block
            current_mem[selected_block] -= processes[p_idx]

    print("\n=== Best Fit Allocation ===")
    print("Process ID\tProcess Size\tBlock Index")
    for idx in range(num_proc):
        dest = f"B{allocation_map[idx] + 1}" if allocation_map[idx] != -1 else "Unallocated"
        print(f"P{idx + 1}\t\t{processes[idx]}\t\t{dest}")

def run_worst_fit(blocks, processes):
    num_blocks = len(blocks)
    num_proc = len(processes)

    current_mem = blocks.copy()
    allocation_map = [-1] * num_proc

    for p_idx in range(num_proc):
        selected_block = -1
        for b_idx in range(num_blocks):
            if current_mem[b_idx] >= processes[p_idx]:
                if selected_block == -1 or current_mem[b_idx] > current_mem[selected_block]:
                    selected_block = b_idx

        if selected_block != -1:
            allocation_map[p_idx] = selected_block
            current_mem[selected_block] -= processes[p_idx]

    print("\n=== Worst Fit Allocation ===")
    print("Process ID\tProcess Size\tBlock Index")
    for idx in range(num_proc):
        dest = f"B{allocation_map[idx] + 1}" if allocation_map[idx] != -1 else "Unallocated"
        print(f"P{idx + 1}\t\t{processes[idx]}\t\t{dest}")

if __name__ == "__main__":
    blocks_count = int(input("Enter total memory blocks count: "))
    if not (1 <= blocks_count <= UPPER_LIMIT):
        print(f"Boundary error: Range allowed is 1 to {UPPER_LIMIT}")
        raise SystemExit

    blocks_list = list(map(int, input("Enter capacity values for memory blocks: ").split()))
    if len(blocks_list) != blocks_count or any(size <= 0 for size in blocks_list):
        print("Invalid dimension entries detected.")
        raise SystemExit

    processes_count = int(input("Enter processes quantity: "))
    if not (1 <= processes_count <= UPPER_LIMIT):
        print(f"Boundary error: Range allowed is 1 to {UPPER_LIMIT}")
        raise SystemExit

    processes_list = list(map(int, input("Enter space requirements for individual processes: ").split()))
    if len(processes_list) != processes_count or any(size <= 0 for size in processes_list):
        print("Invalid resource parameters detected.")
        raise SystemExit

    run_first_fit(blocks_list, processes_list)
    run_best_fit(blocks_list, processes_list)
    run_worst_fit(blocks_list, processes_list)

Enter number of memory blocks: 5
Enter sizes of blocks: 110 520 200 340 620
Enter number of processes: 4
Enter sizes of processes: 212 412 112 426

--- First Fit ---
Process No.	Process Size	Block No.
P1		212		B2
P2		412		B5
P3		112		B2
P4		426		Not Allocated

--- Best Fit ---
Process No.	Process Size	Block No.
P1		212		B4
P2		412		B2
P3		112		B4
P4		426		B5

--- Worst Fit ---
Process No.	Process Size	Block No.
P1		212		B5
P2		412		B2
P3		112		B5
P4		426		Not Allocated


**Program 9:**
Page Replacement Algorithms — FIFO, LRU and Optimal


In [ ]:
# AAKARSHIT YADAV 2472121
MAX_FRAMES_ALLOWED = 10
MAX_PAGES_ALLOWED = 50

def check_page_presence(frame_list, num_frames, target_page):
    for i in range(num_frames):
        if frame_list[i] == target_page:
            return True
    return False

# FIFO Scheduling Logic
def run_fifo(page_stream, total_frames):
    active_frames = [-1] * total_frames
    insert_ptr = 0
    page_faults = 0

    for p in page_stream:
        if not check_page_presence(active_frames, total_frames, p):
            active_frames[insert_ptr] = p
            insert_ptr = (insert_ptr + 1) % total_frames
            page_faults += 1

    print("FIFO System Fault Count =", page_faults)

# LRU Scheduling Logic
def run_lru(page_stream, total_frames):
    active_frames = [-1] * total_frames
    last_accessed_time = [0] * total_frames
    allocated_count = 0
    page_faults = 0

    for current_tick, p in enumerate(page_stream):
        found_idx = -1
        for f_idx in range(total_frames):
            if active_frames[f_idx] == p:
                found_idx = f_idx
                break

        if found_idx == -1:
            page_faults += 1
            if allocated_count < total_frames:
                active_frames[allocated_count] = p
                last_accessed_time[allocated_count] = current_tick
                allocated_count += 1
            else:
                least_recent_idx = 0
                for f_idx in range(1, total_frames):
                    if last_accessed_time[f_idx] < last_accessed_time[least_recent_idx]:
                        least_recent_idx = f_idx
                active_frames[least_recent_idx] = p
                last_accessed_time[least_recent_idx] = current_tick
        else:
            last_accessed_time[found_idx] = current_tick

    print("LRU System Fault Count  =", page_faults)

# Optimal Scheduling Logic
def run_optimal(page_stream, total_frames):
    active_frames = [-1] * total_frames
    allocated_count = 0
    page_faults = 0

    for current_tick, p in enumerate(page_stream):
        if check_page_presence(active_frames, allocated_count, p):
            continue

        page_faults += 1
        if allocated_count < total_frames:
            active_frames[allocated_count] = p
            allocated_count += 1
            continue

        optimal_replacement_idx = -1
        furthest_tick = -1

        for f_idx in range(total_frames):
            try:
                next_occ = page_stream.index(active_frames[f_idx], current_tick + 1)
            except ValueError:
                optimal_replacement_idx = f_idx
                break

            if next_occ > furthest_tick:
                furthest_tick = next_occ
                optimal_replacement_idx = f_idx

        active_frames[optimal_replacement_idx] = p

    print("Optimal System Faults   =", page_faults)

if __name__ == "__main__":
    n_pages = int(input("Enter total pages inside reference string: "))
    if not (1 <= n_pages <= MAX_PAGES_ALLOWED):
        print(f"Boundary error: Total page count range is 1 to {MAX_PAGES_ALLOWED}")
        raise SystemExit

    input_stream = list(map(int, input("Input sequence parameters: ").split()))
    if len(input_stream) != n_pages or any(page < 0 for page in input_stream):
        print(f"Invalid page specifications detected. Input exactly {n_pages} positive numbers.")
        raise SystemExit

    n_frames = int(input("Enter total allocated frames: "))
    if not (1 <= n_frames <= MAX_FRAMES_ALLOWED):
        print(f"Boundary error: Frame allocations range is 1 to {MAX_FRAMES_ALLOWED}")
        raise SystemExit

    run_fifo(input_stream, n_frames)
    run_lru(input_stream, n_frames)
    run_optimal(input_stream, n_frames)

Enter number of pages in reference string: 12
Enter the reference string: 7 0 1 2 0 3 0 4 2 3 0 3
Enter number of frames: 3
FIFO Page Faults  = 10
LRU Page Faults   = 9
Optimal Page Faults = 7


**Program 10:**
Disk Scheduling Algorithms — FCFS, SSTF, SCAN and C-SCAN


In [ ]:
# AAKARSHIT YADAV 2472121
MAX_REQUESTS_LIMIT = 50

def calculate_offset(x, y):
    return abs(x - y)

def execute_fcfs_disk(requests, num_requests, start_head):
    accumulated_motion = 0
    pointer = start_head

    for idx in range(num_requests):
        accumulated_motion += calculate_offset(pointer, requests[idx])
        pointer = requests[idx]

    print("FCFS total displacement   =", accumulated_motion)

def execute_sstf_disk(requests, num_requests, start_head):
    is_resolved = [False] * num_requests
    accumulated_motion = 0
    pointer = start_head

    for _ in range(num_requests):
        nearest_idx = -1
        shortest_seek = float('inf')

        for idx in range(num_requests):
            if not is_resolved[idx]:
                current_seek = calculate_offset(pointer, requests[idx])
                if current_seek < shortest_seek:
                    shortest_seek = current_seek
                    nearest_idx = idx

        is_resolved[nearest_idx] = True
        accumulated_motion += shortest_seek
        pointer = requests[nearest_idx]

    print("SSTF total displacement   =", accumulated_motion)

def execute_scan_disk(requests, num_requests, start_head, disk_limit):
    working_sequence = requests.copy()
    working_sequence.append(start_head)
    working_sequence.sort()

    pivot_idx = working_sequence.index(start_head)
    accumulated_motion = 0
    pointer = start_head

    # Upward sweeping loop
    for idx in range(pivot_idx + 1, len(working_sequence)):
        accumulated_motion += calculate_offset(pointer, working_sequence[idx])
        pointer = working_sequence[idx]

    # Reach outer disk boundary
    accumulated_motion += calculate_offset(pointer, disk_limit - 1)
    pointer = disk_limit - 1

    # Downward sweeping loop
    for idx in range(pivot_idx - 1, -1, -1):
        accumulated_motion += calculate_offset(pointer, working_sequence[idx])
        pointer = working_sequence[idx]

    print("SCAN total displacement   =", accumulated_motion)

def execute_cscan_disk(requests, num_requests, start_head, disk_limit):
    working_sequence = requests.copy()
    working_sequence.append(start_head)
    working_sequence.sort()

    pivot_idx = working_sequence.index(start_head)
    accumulated_motion = 0
    pointer = start_head

    # Forward sweeping loop
    for idx in range(pivot_idx + 1, len(working_sequence)):
        accumulated_motion += calculate_offset(pointer, working_sequence[idx])
        pointer = working_sequence[idx]

    # Reach outer limit
    accumulated_motion += calculate_offset(pointer, disk_limit - 1)

    # Rapid traverse cycle reset
    accumulated_motion += disk_limit - 1
    pointer = 0

    # Traverse loop from base start
    for idx in range(pivot_idx):
        accumulated_motion += calculate_offset(pointer, working_sequence[idx])
        pointer = working_sequence[idx]

    print("C-SCAN total displacement =", accumulated_motion)

if __name__ == "__main__":
    req_count = int(input("Enter disk requests count: "))
    if not (1 <= req_count <= MAX_REQUESTS_LIMIT):
        print(f"Boundary error: total count ranges from 1 to {MAX_REQUESTS_LIMIT}.")
        raise SystemExit

    requests_vector = list(map(int, input("Provide sequence values separated by space: ").split()))
    if len(requests_vector) != req_count:
        print("Length error: input sequence size doesn't match total requested value count.")
        raise SystemExit

    starting_position = int(input("Provide initial head coordinate: "))
    boundary_size = int(input("Provide system cylinder capacity (Max size limit): "))

    if boundary_size < 1 or not (0 <= starting_position < boundary_size):
        print("Invalid boundary configurations or start position defined.")
        raise SystemExit

    if any(req < 0 or req >= boundary_size for req in requests_vector):
        print(f"Error: All requested entries must lie within bounds [0, {boundary_size - 1}].")
        raise SystemExit

    execute_fcfs_disk(requests_vector, req_count, starting_position)
    execute_sstf_disk(requests_vector, req_count, starting_position)
    execute_scan_disk(requests_vector, req_count, starting_position, boundary_size)
    execute_cscan_disk(requests_vector, req_count, starting_position, boundary_size)

Enter number of requests: 8
Enter the requests: 90 180 37 120 14 114 65 67
Enter initial head position: 55
Enter disk size (max cylinder + 1): 200
FCFS   total head movement = 608
SSTF   total head movement = 291
SCAN   total head movement = 329
C-SCAN total head movement = 380


**Program 11:**
File Allocation Methods — Contiguous, Linked and Indexed Allocation


In [ ]:
# AAKARSHIT YADAV 2472121
TOTAL_BLOCKS = 30

system_disk = [0] * TOTAL_BLOCKS  # 0 indicates empty sector, 1 indicates occupied sector

def format_storage():
    global system_disk
    system_disk = [0] * TOTAL_BLOCKS

def run_contiguous_allocation():
    format_storage()
    print("\n--- Contiguous File System Simulation ---")
    files_count = int(input("Enter total file creation limit: "))

    for idx in range(files_count):
        starting_sector, sector_len = map(
            int, input(f"File {idx + 1} -> Enter start block address & length: ").split()
        )

        is_allocatable = True

        if starting_sector < 0 or sector_len <= 0 or starting_sector + sector_len > TOTAL_BLOCKS:
            is_allocatable = False
        else:
            for sector in range(starting_sector, starting_sector + sector_len):
                if system_disk[sector] == 1:
                    is_allocatable = False
                    break

        if not is_allocatable:
            print("Storage failure: Blocks unavailable or target region out of index range.")
            continue

        for sector in range(starting_sector, starting_sector + sector_len):
            system_disk[sector] = 1

        print(f"File {idx + 1} allocated contiguously: Blocks {starting_sector} to {starting_sector + sector_len - 1}")

def run_linked_allocation():
    format_storage()
    print("\n--- Linked File System Simulation ---")
    files_count = int(input("Enter total file creation limit: "))

    for idx in range(files_count):
        block_demand = int(input(f"File {idx + 1} -> Enter blocks required: "))

        if block_demand <= 0:
            print("Invalid input requirement.")
            continue

        chain = []
        for sector in range(TOTAL_BLOCKS):
            if system_disk[sector] == 0 and len(chain) < block_demand:
                system_disk[sector] = 1
                chain.append(sector)

        print(f"File {idx + 1} allocation sequence: ", end="")
        if chain:
            print(" -> ".join(map(str, chain)) + " -> ", end="")
        print("(NULL)")

        if len(chain) < block_demand:
            print("Storage Warning: Disk space exhausted. Partial allocation completed.")

def run_indexed_allocation():
    format_storage()
    print("\n--- Indexed File System Simulation ---")
    files_count = int(input("Enter total file creation limit: "))

    for idx in range(files_count):
        block_demand = int(input(f"File {idx + 1} -> Enter data blocks required: "))

        if block_demand < 0:
            print("Invalid input requirement.")
            continue

        index_table_pointer = -1
        for sector in range(TOTAL_BLOCKS):
            if system_disk[sector] == 0:
                index_table_pointer = sector
                system_disk[sector] = 1
                break

        if index_table_pointer == -1:
            print(f"File {idx + 1}: Simulation interrupted. Index node block unavailable.")
            continue

        allocated_sectors = []
        for sector in range(TOTAL_BLOCKS):
            if system_disk[sector] == 0 and len(allocated_sectors) < block_demand:
                system_disk[sector] = 1
                allocated_sectors.append(sector)

        print(f"File {idx + 1} -> Index block pointer: {index_table_pointer}, Allocated sectors: ", end="")
        print(*allocated_sectors)

        if len(allocated_sectors) < block_demand:
            print("Storage Warning: Disk space exhausted. Partial data indexing completed.")

if __name__ == "__main__":
    print("Main Control Terminal:")
    print("1 -> Contiguous Allocation")
    print("2 -> Linked Allocation")
    print("3 -> Indexed Allocation")

    menu_choice = int(input("Select Scheme Choice (1-3): "))

    if menu_choice == 1:
        run_contiguous_allocation()
    elif menu_choice == 2:
        run_linked_allocation()
    elif menu_choice == 3:
        run_indexed_allocation()
    else:
        print("Selected ID falls outside expected bounds!")

Menu:
1. Contiguous Allocation
2. Linked Allocation
3. Indexed Allocation
Enter choice: 2

--- Linked Allocation ---
Enter number of files: 2
File 1 - number of blocks: 3
File 1 allocated blocks: 0 -> 1 -> 2 -> (NULL)
File 2 - number of blocks: 4
File 2 allocated blocks: 3 -> 4 -> 5 -> 6 -> (NULL)
